# LeetCode #90 — Subsets II: sinh tập con khi có phần tử trùng

**Mức độ:** Medium · **Chủ đề:** Array, Backtracking, Bit Manipulation

Skeleton trên LeetCode:

```python
class Solution:
    def subsetsWithDup(self, nums: List[int]) -> List[List[int]]:
```

Đây là #78 Subsets, nhưng `nums` **có thể có phần tử trùng**. Mục tiêu của notebook:

1. Thấy vì sao cách của #78 sinh ra tập con **trùng lặp**.
2. Kỹ thuật chuẩn: **sắp xếp** + bỏ qua `nums[i] == nums[i-1]` **ở cùng một tầng** (`i > start`).
3. Cách lặp: chỉ mở rộng phần **vừa thêm** khi gặp số trùng.
4. Cách đếm: chọn **bao nhiêu bản** của mỗi giá trị.

# 1. Bài toán là gì?

Cho mảng `nums` (có thể có phần tử trùng), trả về **mọi tập con** (power set). Kết quả **không được** chứa tập con trùng. Thứ tự tùy ý.

**Constraints (theo đề):**

- `1 <= nums.length <= 10`
- `-10 <= nums[i] <= 10`

---

## Ví dụ

```text
nums = [1, 2, 2]  ->  [[], [1], [1,2], [1,2,2], [2], [2,2]]
nums = [0]        ->  [[], [0]]
```

Chú ý `[1, 2]` chỉ xuất hiện **một lần**, dù có hai cách chọn số `2`.

# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Quay lui (backtracking)
2. Sắp xếp (sorting)

Cuối mục có phần **2.3 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Quay lui (backtracking)

### Quay lui là gì?

Quay lui là cách **liệt kê / tìm kiếm có hệ thống** mọi lời giải bằng cách xây lời giải **từng bước một**. Ở mỗi bước ta thử một lựa chọn, đi tiếp; nếu đi hết (hoặc thấy không thể thành lời giải) thì **hoàn tác** lựa chọn đó và thử lựa chọn khác.

Hình dung bằng **cây quyết định**: mỗi nút là một trạng thái "đã chọn được một phần", mỗi cạnh là một lựa chọn, mỗi lá là một lời giải hoàn chỉnh. Quay lui = DFS trên cây này.

```text
Liệt kê mọi dãy độ dài 2 từ {a, b}:
                 []
            /          \
         [a]            [b]
        /   \          /   \
    [a,a] [a,b]    [b,a] [b,b]      ← 4 lá = 4 lời giải
```

### Khung chuẩn: chọn → đi tiếp → bỏ chọn

```text
def backtrack(trạng_thái):
    if trạng_thái là lời giải hoàn chỉnh:
        ghi lại BẢN SAO
        return
    for lựa_chọn in các lựa chọn hợp lệ:
        chọn          (thêm vào trạng thái)
        backtrack(trạng_thái)
        bỏ chọn       (hoàn tác — trả trạng thái về như cũ)
```

In [1]:
def moi_day(chu, k):
    out, path = [], []
    def bt(sau=0):
        if len(path) == k:
            out.append("".join(path))           # ghi bản sao
            return
        for ch in chu:
            path.append(ch)                     # chọn
            bt(sau + 1)                         # đi tiếp
            path.pop()                          # bỏ chọn
    bt()
    return out

print(moi_day("ab", 2))
print("số dãy độ dài 3 từ 'abc':", len(moi_day("abc", 3)), "= 3^3")

['aa', 'ab', 'ba', 'bb']
số dãy độ dài 3 từ 'abc': 27 = 3^3


### Ba dạng kinh điển

| Dạng | Cách chọn ở mỗi bước | Kích thước kết quả |
|---|---|---|
| **Tập con** (subset) | mỗi phần tử: lấy hoặc không; hoặc chọn phần tử tiếp theo từ chỉ số `start` | `2^n` |
| **Tổ hợp** chập `k` | chọn phần tử từ `start` trở đi (không quay lại phía trước) | `C(n, k)` |
| **Hoán vị** | chọn phần tử **chưa dùng** (mảng `used`) | `n!` |

Tham số `start` đảm bảo không sinh lại cùng một tập theo thứ tự khác (`{1,2}` và `{2,1}` là một).

In [2]:
def tap_con(nums):
    out, path = [], []
    def bt(start):
        out.append(path[:])                     # mọi nút của cây đều là một tập con
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1)
            path.pop()
    bt(0)
    return out

def hoan_vi(nums):
    out, path, used = [], [], [False] * len(nums)
    def bt():
        if len(path) == len(nums):
            out.append(path[:]); return
        for i in range(len(nums)):
            if used[i]:
                continue
            used[i] = True; path.append(nums[i])
            bt()
            path.pop(); used[i] = False
    bt()
    return out

print("tập con của [1,2,3]:", tap_con([1, 2, 3]))
print("hoán vị của [1,2,3]:", hoan_vi([1, 2, 3]))

tập con của [1,2,3]: [[], [1], [1, 2], [1, 2, 3], [1, 3], [2], [2, 3], [3]]
hoán vị của [1,2,3]: [[1, 2, 3], [1, 3, 2], [2, 1, 3], [2, 3, 1], [3, 1, 2], [3, 2, 1]]


### Vết chạy: thấy rõ "chọn / bỏ chọn"

In [3]:
def vet_tap_con(nums):
    path = []
    def bt(start, sau):
        print("    " * sau + f"path = {path}")
        for i in range(start, len(nums)):
            path.append(nums[i])
            bt(i + 1, sau + 1)
            path.pop()
    bt(0, 0)

vet_tap_con([1, 2, 3])

path = []
    path = [1]
        path = [1, 2]
            path = [1, 2, 3]
        path = [1, 3]
    path = [2]
        path = [2, 3]
    path = [3]


### Cắt tỉa (pruning) và bỏ trùng

- **Cắt tỉa**: nếu trạng thái hiện tại **chắc chắn** không dẫn tới lời giải (tổng đã vượt target, còn quá ít phần tử…), `return` sớm — bỏ cả nhánh cây.
- **Bỏ trùng** khi đầu vào có phần tử lặp: **sắp xếp** trước, rồi trong cùng một tầng (cùng vòng `for`) bỏ qua phần tử bằng phần tử ngay trước nó: `if i > start and nums[i] == nums[i-1]: continue`.

In [4]:
def tap_con_tong(nums, target):
    nums = sorted(nums)
    out, path, dem = [], [], [0]
    def bt(start, con_lai):
        dem[0] += 1
        if con_lai == 0:
            out.append(path[:]); return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue                        # bỏ trùng cùng tầng
            if nums[i] > con_lai:
                break                           # cắt tỉa: đã sắp xếp nên các số sau còn lớn hơn
            path.append(nums[i])
            bt(i + 1, con_lai - nums[i])
            path.pop()
    bt(0, target)
    return out, dem[0]

print("các tập có tổng 8 từ [10,1,2,7,6,1,5]:", tap_con_tong([10, 1, 2, 7, 6, 1, 5], 8))

các tập có tổng 8 từ [10,1,2,7,6,1,5]: ([[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]], 17)


### Lỗi hay gặp

- `out.append(path)` thay vì `out.append(path[:])` → mọi phần tử của `out` là **cùng một** list, cuối cùng đều rỗng.
- Quên **bỏ chọn** (`pop`, `used[i] = False`) → trạng thái "dính" sang nhánh khác.
- Dùng `start = i` thay vì `i + 1` (hoặc ngược lại) → lấy lặp phần tử ngoài ý muốn / thiếu lời giải.
- Bỏ trùng bằng `i > 0` thay vì `i > start` → bỏ mất lời giải hợp lệ dùng hai phần tử bằng nhau ở hai tầng khác nhau.

In [5]:
def sai_khong_sao_chep(nums):
    out, path = [], []
    def bt(start):
        out.append(path)                        # SAI: thêm chính list path
        for i in range(start, len(nums)):
            path.append(nums[i]); bt(i + 1); path.pop()
    bt(0)
    return out
print("quên sao chép:", sai_khong_sao_chep([1, 2]))

quên sao chép: [[], [], [], []]


## 2.2 Sắp xếp (sorting)

### Sắp xếp trong Python

- `sorted(a)` trả về **list mới**; `a.sort()` sắp xếp **tại chỗ** (trả về `None`).
- `key=` chọn tiêu chí: `sorted(words, key=len)`, `sorted(pairs, key=lambda p: p[1])`.
- `reverse=True` để giảm dần; sắp xếp theo nhiều tiêu chí bằng `key` trả về tuple.
- Thuật toán bên trong là **Timsort**: `O(n log n)`, **ổn định** (stable — hai phần tử bằng nhau giữ nguyên thứ tự ban đầu).

In [6]:
a_sort = [5, 2, 9, 1, 5, 6]
print("sorted:", sorted(a_sort), "| gốc không đổi:", a_sort)
a_sort.sort(reverse=True)
print("sort giảm dần tại chỗ:", a_sort)

nguoi = [("An", 25), ("Bình", 20), ("Chi", 25), ("Dũng", 20)]
print("theo tuổi (ổn định — An trước Chi):", sorted(nguoi, key=lambda p: p[1]))
print("tuổi giảm, rồi tên tăng:", sorted(nguoi, key=lambda p: (-p[1], p[0])))
print("chuỗi sắp theo ký tự:", "".join(sorted("banana")))

sorted: [1, 2, 5, 5, 6, 9] | gốc không đổi: [5, 2, 9, 1, 5, 6]
sort giảm dần tại chỗ: [9, 6, 5, 5, 2, 1]
theo tuổi (ổn định — An trước Chi): [('Bình', 20), ('Dũng', 20), ('An', 25), ('Chi', 25)]
tuổi giảm, rồi tên tăng: [('An', 25), ('Chi', 25), ('Bình', 20), ('Dũng', 20)]
chuỗi sắp theo ký tự: aaabnn


### Vì sao sắp xếp hay là bước đầu tiên?

Sau khi sắp xếp: phần tử **bằng nhau đứng cạnh nhau** (dễ bỏ trùng), có thể dùng **hai con trỏ** / **tìm nhị phân**, các khoảng chồng nhau **liền kề** nhau. Đổi lại tốn `O(n log n)`.

### Sắp xếp chèn (insertion sort)

Giống xếp bài trên tay: lấy lần lượt từng phần tử, **chèn** vào đúng vị trí trong phần đã sắp xếp bên trái. `O(n²)` xấu nhất, nhưng `O(n)` khi dữ liệu gần như đã sắp xếp.

In [7]:
def sap_xep_chen(a):
    a = a[:]
    for i in range(1, len(a)):
        x, j = a[i], i - 1
        while j >= 0 and a[j] > x:
            a[j + 1] = a[j]            # dịch phần tử lớn hơn sang phải
            j -= 1
        a[j + 1] = x
        print(f"   sau khi chèn {x}: {a[:i + 1]} | {a[i + 1:]}")
    return a

print("→", sap_xep_chen([5, 2, 4, 6, 1, 3]))

   sau khi chèn 2: [2, 5] | [4, 6, 1, 3]
   sau khi chèn 4: [2, 4, 5] | [6, 1, 3]
   sau khi chèn 6: [2, 4, 5, 6] | [1, 3]
   sau khi chèn 1: [1, 2, 4, 5, 6] | [3]
   sau khi chèn 3: [1, 2, 3, 4, 5, 6] | []
→ [1, 2, 3, 4, 5, 6]


### Sắp xếp trộn (merge sort) và phép trộn hai dãy đã sắp xếp

**Chia để trị**: chia đôi, sắp xếp mỗi nửa (đệ quy), rồi **trộn** hai nửa đã sắp xếp. Trộn: hai con trỏ ở đầu hai dãy, mỗi bước lấy phần tử nhỏ hơn. `O(n log n)` mọi trường hợp, ổn định (nhờ lấy bên trái khi bằng nhau).

```text
[5,2,4,6,1,3] → [5,2,4] [6,1,3] → [5] [2,4] [6] [1,3] → … → [2,4,5] [1,3,6] → [1,2,3,4,5,6]
```

In [8]:
def tron(x, y):
    i = j = 0
    out = []
    while i < len(x) and j < len(y):
        if x[i] <= y[j]:               # <= giữ ổn định
            out.append(x[i]); i += 1
        else:
            out.append(y[j]); j += 1
    out.extend(x[i:]); out.extend(y[j:])
    return out

def sap_xep_tron(a, sau=0):
    if len(a) <= 1:
        return a
    mid = len(a) // 2
    trai, phai = sap_xep_tron(a[:mid], sau + 1), sap_xep_tron(a[mid:], sau + 1)
    kq = tron(trai, phai)
    print("    " * sau + f"trộn {trai} + {phai} = {kq}")
    return kq

print("→", sap_xep_tron([5, 2, 4, 6, 1, 3]))

        trộn [2] + [4] = [2, 4]
    trộn [5] + [2, 4] = [2, 4, 5]
        trộn [1] + [3] = [1, 3]
    trộn [6] + [1, 3] = [1, 3, 6]
trộn [2, 4, 5] + [1, 3, 6] = [1, 2, 3, 4, 5, 6]
→ [1, 2, 3, 4, 5, 6]


### Bảng so sánh

| Thuật toán | Tốt nhất | Xấu nhất | Bộ nhớ phụ | Ổn định |
|---|---|---|---|---|
| Insertion sort | `O(n)` | `O(n²)` | `O(1)` | có |
| Merge sort | `O(n log n)` | `O(n log n)` | `O(n)` (mảng) | có |
| Quick sort | `O(n log n)` | `O(n²)` | `O(log n)` | không |
| Timsort (`sorted`) | `O(n)` | `O(n log n)` | `O(n)` | có |

Lỗi hay gặp: `a = a.sort()` (gán `None`); quên rằng `sorted` tạo list mới (tốn bộ nhớ); `key` trả về kiểu không so sánh được.

## 2.3 Áp dụng vào bài này

Mọi **tập con** khi có trùng: **sắp xếp** rồi quay lui như #78, bỏ trùng cùng tầng bằng `i > start and nums[i] == nums[i−1]`.

# 3. Vấn đề: dùng nguyên code #78

In [9]:
from typing import List
from collections import Counter
import itertools, random

def subsets_78(nums: List[int]) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

print(subsets_78([1, 2, 2]))

[[], [1], [1, 2], [1, 2, 2], [1, 2], [2], [2, 2], [2]]


`[1, 2]` và `[2]` xuất hiện **hai lần** — vì có hai số `2` ở hai vị trí khác nhau.

## Baseline: sinh hết rồi khử trùng bằng `set`

In [10]:
def subsets_dedup_set(nums: List[int]) -> List[List[int]]:
    seen = set()
    for s in subsets_78(sorted(nums)):          # sắp xếp để [1,2] và [2,1] cùng dạng
        seen.add(tuple(s))
    return [list(t) for t in seen]

print(sorted(subsets_dedup_set([1, 2, 2])))

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


Đúng, nhưng vẫn sinh ra **toàn bộ** `2^n` tập con rồi mới loại — lãng phí khi có nhiều phần tử trùng (vd. `[1] * 10` chỉ có 11 tập con khác nhau, nhưng sinh 1024). Ta muốn **không bao giờ sinh** bản trùng.

# 4. Ý tưởng chính: sắp xếp + bỏ trùng ở cùng tầng

**Sắp xếp** để các số giống nhau đứng **liền nhau**.

Trong cây backtracking của #78, mỗi **tầng** (một lời gọi `backtrack(start)`) chọn **phần tử tiếp theo** của tập con. Nếu ở cùng một tầng ta đã thử chọn số `2` (ở vị trí `i−1`), thì thử chọn số `2` khác (ở vị trí `i`) sẽ dẫn tới **cùng các tập con** → bỏ qua:

```text
for i in range(start, n):
    if i > start and nums[i] == nums[i-1]:  continue    # trùng ở cùng tầng
    ...
```

`i > start` (không phải `i > 0`): cho phép chọn số `2` thứ hai khi **đi sâu** (tầng sau) — để có `[2, 2]` — nhưng không cho chọn lại ở **cùng tầng**.

In [11]:
def subsets_with_dup(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

print(subsets_with_dup([1, 2, 2]))
print(subsets_with_dup([2, 1, 2]))

[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
[[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]


# 5. Mô phỏng: cây quyết định với nhánh bị cắt

In [12]:
def trace_subsets_dup(nums: List[int]):
    nums = sorted(nums)
    result, path = [], []
    print("nums sau sắp xếp:", nums)
    def backtrack(start, depth):
        ind = "  " * depth
        print(f"{ind}lưu {path}")
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                print(f"{ind}  bỏ qua nums[{i}] = {nums[i]} (trùng nums[{i-1}] ở cùng tầng)")
                continue
            path.append(nums[i])
            backtrack(i + 1, depth + 1)
            path.pop()
    backtrack(0, 0)
    return result

_ = trace_subsets_dup([1, 2, 2])

nums sau sắp xếp: [1, 2, 2]
lưu []
  lưu [1]
    lưu [1, 2]
      lưu [1, 2, 2]
    bỏ qua nums[2] = 2 (trùng nums[1] ở cùng tầng)
  lưu [2]
    lưu [2, 2]
  bỏ qua nums[2] = 2 (trùng nums[1] ở cùng tầng)


Ở tầng gốc (`start = 0`), `nums[2] = 2` bị bỏ vì đã thử `nums[1] = 2`. Nhưng ở tầng `[2]` (`start = 2`), `nums[2]` được chọn vì `i == start` → sinh ra `[2, 2]`.

# 6. Cách 2: Lặp — chỉ mở rộng phần vừa thêm

Cách lặp của #78: mỗi số `x` nhân đôi danh sách. Với số **trùng** với số trước, nếu nối `x` vào **mọi** tập con sẽ tạo bản trùng. Chỉ nối vào các tập con **vừa được tạo ở bước trước** (những tập đã chứa bản `x` trước đó).

```text
sorted = [1, 2, 2]
[]
[] [1]                      thêm 1 vào tất cả
[] [1] [2] [1,2]            thêm 2 vào tất cả (2 là số mới)  -> phần mới: [2], [1,2]
[] [1] [2] [1,2] [2,2] [1,2,2]   2 trùng -> chỉ thêm vào phần mới: [2], [1,2]
```

In [13]:
def subsets_with_dup_iter(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result = [[]]
    new_start = 0
    for i, x in enumerate(nums):
        start = new_start if i > 0 and x == nums[i - 1] else 0
        new_start = len(result)
        result += [s + [x] for s in result[start:]]
    return result

print(subsets_with_dup_iter([1, 2, 2]))

[[], [1], [2], [1, 2], [2, 2], [1, 2, 2]]


# 7. Cách 3: Đếm — chọn bao nhiêu bản của mỗi giá trị

Với giá trị `v` xuất hiện `c` lần, một tập con chứa `0, 1, …, c` bản của `v`. Tổng số tập con khác nhau là **tích** `∏ (c_v + 1)`.

In [14]:
def subsets_with_dup_count(nums: List[int]) -> List[List[int]]:
    result = [[]]
    for v, c in sorted(Counter(nums).items()):
        result = [s + [v] * k for s in result for k in range(c + 1)]
    return result

print(subsets_with_dup_count([1, 2, 2]))
print("Số tập con của [1]*10:", len(subsets_with_dup_count([1] * 10)), "| [1,2,2,3,3,3]:", len(subsets_with_dup_count([1, 2, 2, 3, 3, 3])), "= 2·3·4")

[[], [2], [2, 2], [1], [1, 2], [1, 2, 2]]
Số tập con của [1]*10: 11 | [1,2,2,3,3,3]: 24 = 2·3·4


# 8. Phiên bản đúng format LeetCode

In [15]:
class Solution:
    def subsetsWithDup(self, nums: List[int]) -> List[List[int]]:
        nums.sort()
        result, path = [], []

        def backtrack(start: int) -> None:
            result.append(path[:])
            for i in range(start, len(nums)):
                if i > start and nums[i] == nums[i - 1]:
                    continue
                path.append(nums[i])
                backtrack(i + 1)
                path.pop()

        backtrack(0)
        return result

In [16]:
def norm(res):
    return sorted(sorted(s) for s in res)

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<16} PASS  {actual}")


sol = Solution()
check("Ví dụ 1", norm(sol.subsetsWithDup([1, 2, 2])), [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]])
check("Ví dụ 2", norm(sol.subsetsWithDup([0])), [[], [0]])
check("Toàn trùng", norm(sol.subsetsWithDup([3, 3, 3])), [[], [3], [3, 3], [3, 3, 3]])
check("Chưa sắp xếp", norm(sol.subsetsWithDup([4, 1, 4])), [[], [1], [1, 4], [1, 4, 4], [4], [4, 4]])

random.seed(90)
for _ in range(2000):
    nums = [random.randint(-2, 2) for _ in range(random.randint(1, 9))]
    expected = norm(subsets_dedup_set(nums))
    for f in (sol.subsetsWithDup, subsets_with_dup, subsets_with_dup_iter, subsets_with_dup_count):
        res = f(nums[:])
        assert len(res) == len(expected), (f.__name__, nums)          # không có bản trùng
        assert norm(res) == expected
print("2000 mảng ngẫu nhiên: 4 cách cho đúng tập kết quả, không bản trùng")

Ví dụ 1          PASS  [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
Ví dụ 2          PASS  [[], [0]]
Toàn trùng       PASS  [[], [3], [3, 3], [3, 3, 3]]
Chưa sắp xếp     PASS  [[], [1], [1, 4], [1, 4, 4], [4], [4, 4]]


2000 mảng ngẫu nhiên: 4 cách cho đúng tập kết quả, không bản trùng


# 9. Vì sao đúng? Độ phức tạp?

**Không trùng:** mỗi tập con (dạng đã sắp xếp) có một "cách chọn chuẩn": khi cần `j` bản của giá trị `v`, luôn lấy `j` bản **đầu tiên** (vị trí nhỏ nhất) trong cụm `v`. Quy tắc `i > start and nums[i] == nums[i-1] → continue` loại đúng mọi cách chọn **không chuẩn** (chọn bản thứ hai của cụm trong khi bỏ qua bản thứ nhất ở cùng tầng).

**Không thiếu:** cách chọn chuẩn luôn đi qua: trong cụm, ta chỉ chọn vị trí `i` khi `i == start` (đã lấy bản trước đó ở tầng trên) hoặc `nums[i] != nums[i-1]` (đầu cụm).

```text
Time:  O(n · 2^n) trường hợp xấu (không có trùng); O(n log n) cho sắp xếp
Space: O(n) ngoài output
```

# 10. Những lỗi dễ mắc

## Lỗi 1: Quên sắp xếp

In [17]:
def wrong_no_sort(nums: List[int]) -> List[List[int]]:
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

r = wrong_no_sort([2, 1, 2])
print("Sai :", r, "-> có", len(r), "tập con, đúng là", len(sol.subsetsWithDup([2, 1, 2])))

Sai : [[], [2], [2, 1], [2, 1, 2], [2, 2], [1], [1, 2], [2]] -> có 8 tập con, đúng là 6


Hai số `2` không liền nhau nên điều kiện `nums[i] == nums[i-1]` không bắt được: `[2]` và `[2, 1]`/`[1, 2]` bị sinh hai lần.

## Lỗi 2: `i > 0` thay vì `i > start`

In [18]:
def wrong_i_gt_0(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            if i > 0 and nums[i] == nums[i - 1]:       # SAI: chặn luôn việc đi sâu
                continue
            path.append(nums[i])
            backtrack(i + 1)
            path.pop()
    backtrack(0)
    return result

print("Sai :", wrong_i_gt_0([1, 2, 2]), "| thiếu [2, 2] và [1, 2, 2]")

Sai : [[], [1], [1, 2], [2]] | thiếu [2, 2] và [1, 2, 2]


## Lỗi 3: Dùng `set` trên list

`set()` không chứa được `list` (không hash được) → `TypeError`. Phải đổi sang `tuple`. Và nếu không sắp xếp trước, `(1, 2)` và `(2, 1)` là hai tuple khác nhau.

In [19]:
try:
    {[1, 2]}
except TypeError as e:
    print("Lỗi:", e)
print("(1, 2) == (2, 1)?", (1, 2) == (2, 1))

Lỗi: unhashable type: 'list'
(1, 2) == (2, 1)? False


## Lỗi 4: Cách lặp nhưng quên cập nhật `new_start` khi số mới khác số trước

Khi `x` khác số trước, phải mở rộng **toàn bộ** (`start = 0`), và lưu `new_start = len(result)` **trước** khi nối.

# 11. Mẹo nhận diện pattern

**"Input có trùng, output không được trùng"** trong backtracking → **sắp xếp + bỏ qua phần tử giống phần tử trước ở cùng tầng**:

```python
nums.sort()
for i in range(start, n):
    if i > start and nums[i] == nums[i - 1]:
        continue
```

Áp dụng cho: **#40 Combination Sum II**, **#47 Permutations II** (dạng `used[]`: `if i > 0 and nums[i] == nums[i-1] and not used[i-1]`), **#491 Non-decreasing Subsequences** (không sắp xếp được → dùng `set` cục bộ mỗi tầng).

# 12. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Có bao nhiêu tập con khác nhau của `[1, 1, 2, 2, 2]`? Liệt kê theo cách đếm (mục 7).

## Bài 2 — Điền chỗ trống

In [20]:
def exercise_2_subsets_with_dup(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result, path = [], []
    def backtrack(start):
        result.append(path[:])
        for i in range(start, len(nums)):
            # TODO: bỏ trùng ở cùng tầng; chọn -> đệ quy -> bỏ chọn
            pass
    backtrack(0)
    return result


# assert sorted(map(sorted, exercise_2_subsets_with_dup([1, 2, 2]))) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
# assert len(exercise_2_subsets_with_dup([1] * 10)) == 11

## Bài 3 — Biến thể: Combination Sum II (LeetCode #40)

Mỗi số dùng **tối đa một lần**; `candidates` có thể trùng. Tìm mọi tổ hợp khác nhau có tổng `target`.

```python
combination_sum2([10, 1, 2, 7, 6, 1, 5], 8) -> [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
```

In [21]:
def exercise_3_combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_3_combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
# assert sorted(exercise_3_combination_sum2([2, 5, 2, 1, 2], 5)) == [[1, 2, 2], [5]]

## Bài 4 — Biến thể: Permutations II (LeetCode #47)

Mọi hoán vị **khác nhau** của `nums` (có trùng).

```python
permute_unique([1, 1, 2]) -> [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
```

Gợi ý nhanh: sắp xếp; dùng `used[]`; bỏ `i` nếu `nums[i] == nums[i-1]` và `not used[i-1]`.

In [22]:
def exercise_4_permute_unique(nums: List[int]) -> List[List[int]]:
    # TODO
    pass


# assert sorted(exercise_4_permute_unique([1, 1, 2])) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
# assert len(exercise_4_permute_unique([1, 2, 3])) == 6
# assert len(exercise_4_permute_unique([2, 2, 2, 3])) == 4

## Bài 5 — Mini challenge: đếm tập con khác nhau có tổng bằng `target`

Không liệt kê; trả **số lượng** tập con **khác nhau** (theo multiset) có tổng đúng bằng `target`. `nums` gồm số nguyên **dương**, có thể trùng.

```python
count_distinct_subset_sums([1, 2, 2, 3], 5) -> 2      # {2, 3} và {1, 2, 2}
```

Gợi ý nhanh: DP theo **giá trị** (không theo vị trí): với giá trị `v` có `c` bản, `new[s] = Σ_{k=0..c} old[s − k·v]`.

In [23]:
def exercise_5_count_distinct_subset_sums(nums: List[int], target: int) -> int:
    # TODO
    pass


# assert exercise_5_count_distinct_subset_sums([1, 2, 2, 3], 5) == 2
# assert exercise_5_count_distinct_subset_sums([1, 1, 1], 2) == 1
# assert exercise_5_count_distinct_subset_sums([5], 3) == 0

# 13. Gợi ý

**Bài 1**
- Mức 1: `(2 + 1) · (3 + 1) = 12`.

**Bài 3**
- Mức 1: như #90, thêm `remain`; chỉ lưu khi `remain == 0`.
- Mức 2: sắp xếp → `if nums[i] > remain: break`.

**Bài 4**
- Mức 1: `if used[i]: continue`.
- Mức 2: `if i > 0 and nums[i] == nums[i-1] and not used[i-1]: continue` — chỉ được dùng bản sau khi bản trước **đang** được dùng trên đường hiện tại.

**Bài 5**
- Mức 1: `dp = [1] + [0] * target`; với mỗi `(v, c)` trong `Counter(nums)` tính `dp` mới.

# 14. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [24]:
r = subsets_with_dup_count([1, 1, 2, 2, 2])
print(len(r), r)

12 [[], [2], [2, 2], [2, 2, 2], [1], [1, 2], [1, 2, 2], [1, 2, 2, 2], [1, 1], [1, 1, 2], [1, 1, 2, 2], [1, 1, 2, 2, 2]]


## Đáp án Bài 2

In [25]:
def answer_2_subsets_with_dup(nums: List[int]) -> List[List[int]]:
    return subsets_with_dup(nums)

assert sorted(map(sorted, answer_2_subsets_with_dup([1, 2, 2]))) == [[], [1], [1, 2], [1, 2, 2], [2], [2, 2]]
assert len(answer_2_subsets_with_dup([1] * 10)) == 11
print("Bài 2 PASS (giống subsets_with_dup ở mục 4)")

Bài 2 PASS (giống subsets_with_dup ở mục 4)


## Đáp án Bài 3

In [26]:
def answer_3_combination_sum2(candidates: List[int], target: int) -> List[List[int]]:
    nums = sorted(candidates)
    result, path = [], []
    def backtrack(start, remain):
        if remain == 0:
            result.append(path[:])
            return
        for i in range(start, len(nums)):
            if i > start and nums[i] == nums[i - 1]:
                continue
            if nums[i] > remain:
                break
            path.append(nums[i])
            backtrack(i + 1, remain - nums[i])
            path.pop()
    backtrack(0, target)
    return result

assert sorted(answer_3_combination_sum2([10, 1, 2, 7, 6, 1, 5], 8)) == [[1, 1, 6], [1, 2, 5], [1, 7], [2, 6]]
assert sorted(answer_3_combination_sum2([2, 5, 2, 1, 2], 5)) == [[1, 2, 2], [5]]
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [27]:
def answer_4_permute_unique(nums: List[int]) -> List[List[int]]:
    nums = sorted(nums)
    result, path, used = [], [], [False] * len(nums)
    def backtrack():
        if len(path) == len(nums):
            result.append(path[:])
            return
        for i in range(len(nums)):
            if used[i]:
                continue
            if i > 0 and nums[i] == nums[i - 1] and not used[i - 1]:
                continue
            used[i] = True
            path.append(nums[i])
            backtrack()
            path.pop()
            used[i] = False
    backtrack()
    return result

assert sorted(answer_4_permute_unique([1, 1, 2])) == [[1, 1, 2], [1, 2, 1], [2, 1, 1]]
assert len(answer_4_permute_unique([1, 2, 3])) == 6
assert len(answer_4_permute_unique([2, 2, 2, 3])) == 4
for _ in range(300):
    nums = [random.randint(0, 2) for _ in range(random.randint(1, 6))]
    assert sorted(map(tuple, answer_4_permute_unique(nums))) == sorted(set(itertools.permutations(nums)))
print("Bài 4 PASS")

Bài 4 PASS


## Đáp án Bài 5

In [28]:
def answer_5_count_distinct_subset_sums(nums: List[int], target: int) -> int:
    dp = [1] + [0] * target
    for v, c in Counter(nums).items():
        new = [0] * (target + 1)
        for s in range(target + 1):
            for k in range(c + 1):
                if s - k * v < 0:
                    break
                new[s] += dp[s - k * v]
        dp = new
    return dp[target]

assert answer_5_count_distinct_subset_sums([1, 2, 2, 3], 5) == 2
assert answer_5_count_distinct_subset_sums([1, 1, 1], 2) == 1
assert answer_5_count_distinct_subset_sums([5], 3) == 0
for _ in range(1000):
    nums = [random.randint(1, 4) for _ in range(random.randint(1, 8))]
    t = random.randint(0, 12)
    expected = sum(1 for s in subsets_with_dup(nums) if sum(s) == t)
    assert answer_5_count_distinct_subset_sums(nums, t) == expected
print("Bài 5 PASS (khớp liệt kê trên 1000 test)")

Bài 5 PASS (khớp liệt kê trên 1000 test)


# 15. Tổng kết

> Sắp xếp để các số trùng đứng liền nhau; trong vòng `for i in range(start, n)`, bỏ qua `i > start and nums[i] == nums[i-1]`. Mỗi multiset con chỉ được sinh một lần, theo "cách chọn chuẩn".

---

## Checklist trước khi nộp

- [ ] Đã **sắp xếp** `nums`?
- [ ] Điều kiện là `i > start` (không phải `i > 0`)?
- [ ] Lưu bản sao `path[:]`, có tập rỗng?
- [ ] Không dùng `set` để khử trùng sau khi sinh?